In [ ]:
import pandas as pd
import os

# 重新获取股票代码

In [ ]:
id_folder = '/Users/akc/Desktop/PHD/research/NewEnergyAssetPricing2024/dataset/stock_data/stock_id'
id = []

for file in os.listdir(id_folder):
    if file.endswith('.xlsx'):
        file = pd.read_excel(os.path.join(id_folder, file))
        id.append(file)
id = pd.concat(id)

id


In [ ]:
'ssss.ss'.split('.')

In [ ]:
def split_code(x):
    try:
        code = x.split('.')
        if code[1] in ['SZ', 'SH']:
            return code[0]
    except:
        return x

id['stock_code'] = id['证券代码'].apply(split_code)
id.dropna(subset=['stock_code'], inplace=True)

In [ ]:
id['stock_code'].to_csv('/Users/akc/Desktop/PHD/research/NewEnergyAssetPricing2024/dataset/stock_data/stock_id/stock_id2.csv', index=False)

In [ ]:
# 手动将上述生成的文件剔除“数据来源：Wind”
# 将stock_id2.csv中的数据替换至stock_id.txt中

# 根据 stock_id.txt 中的数据，获取目标股票代码

In [ ]:
import pandas as pd
import os
import tqdm
import numpy as np

In [ ]:
id_path = '/Users/akc/Desktop/PHD/research/NewEnergyAssetPricing2024/dataset/stock_data/stock_id/stock_id.txt'
id_list = []
with open(id_path, 'r') as f:
    for line in f:
        id_list.append(line.strip())
id_list

In [ ]:
# 读取单个文件在2min40s 左右
# file_path = '/Users/akc/Desktop/PHD/research/NewEnergyAssetPricing2024/dataset/text_data/TextData/2018/用户与公司问答-2018.xlsx'
# file_data = pd.read_excel(file_path, engine='openpyxl')

In [ ]:
def data_process(file_path):
    file_data = pd.read_excel(file_path, engine='openpyxl')
    colname_rectified = {'Scode':'stock_code', 'Coname':'company_name', 'Qtm':'pubDate', 'Qsubj':'ques_text', 
                         'Wpaonot':'whether_ans', 'Recvtm':'ansDate', 'Reply':'ans_text'}
    cols = colname_rectified.keys()
    file_data = file_data[cols]
    file_data.rename(columns=colname_rectified, inplace=True)
    # 删除第一行
    file_data = file_data.iloc[1:]
    # 筛选出目标股票代码
    file_data = file_data[file_data['stock_code'].isin(id_list)]
    return file_data

# p1 = '/Users/akc/Desktop/PHD/research/NewEnergyAssetPricing2024/dataset/text_data/TextData/2018/用户与公司问答-2018.xlsx'
# data_process(p1)

In [ ]:
file_data = '/Users/akc/Desktop/PHD/research/NewEnergyAssetPricing2024/dataset/text_data/TextData'
folders_name = os.listdir(file_data)
folders_name.remove('.DS_Store')

file_list = []
for folder in tqdm.tqdm(folders_name):
    folder_path = os.path.join(file_data, folder)
    for file in os.listdir(folder_path):
        if file.endswith('.xlsx'):
            file_path = os.path.join(folder_path, file)
            file_list.append(data_process(file_path))

In [ ]:
df = pd.concat(file_list)

df.reset_index(inplace=True, drop=True)
df.to_excel('/Users/akc/Desktop/PHD/research/NewEnergyAssetPricing2024/dataset/text_data/original_data.xlsx', index=False)

In [ ]:
df.shape

In [ ]:
def ifdate(x):
    try:
        pd.to_datetime(x)
        return True
    except:
        return False
df['ifpubDate'] = df['pubDate'].apply(ifdate)
df[df['ifpubDate']==False]


In [ ]:
# 删除不符合日期格式
df = df[df['ifpubDate']==True]
df.shape

In [ ]:
df['pubDate'] = pd.to_datetime(df['pubDate'])

In [ ]:
def ifdate(x):
    try:
        pd.to_datetime(x)
        return True
    except:
        return False
df['ifpubDate'] = df['pubDate'].apply(ifdate)
df[df['ifpubDate']==False]

In [ ]:
df['pubDate'] = df['pubDate'].apply(lambda x: pd.to_datetime(x))

In [ ]:
df.sort_values(by='pubDate')

In [ ]:
def ifdate(x):
    if pd.isnull(x):
        return True
    try:
        pd.to_datetime(x)
        return True
    except:
        return False
df['ifansDate'] = df['ansDate'].apply(ifdate)
df[df['ifansDate']==False]
    

In [ ]:
def convert_ansDate(x):
    try:
        return pd.to_datetime(x)
    except:
        return np.nan
df['ansDate'] = df['ansDate'].apply(convert_ansDate)

In [ ]:
df.sort_values(by='ansDate')

In [ ]:
df.replace(pd.NaT, np.nan, inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
df.sort_values('pubDate', inplace=True)
df.reset_index(inplace=True, drop=True)
# 删除 ifpubDate 和 ifansDate
df.drop(columns=['ifpubDate', 'ifansDate'], inplace=True)
df.to_excel('/Users/akc/Desktop/PHD/research/NewEnergyAssetPricing2024/dataset/text_data/original_data.xlsx', index=False)

# 读入数据

In [ ]:
import pandas as pd
import os

In [ ]:
# 读入16M 的 excel 用时11.2s
ndf = pd.read_excel('/Users/akc/Desktop/PHD/research/NewEnergyAssetPricing2024/dataset/text_data/textWithSimilarity.xlsx', converters={'stock_code': str, 'pubDate': pd.to_datetime, 'ansDate': pd.to_datetime})
ndf.head()

In [ ]:
ndf['pubDate'] = pd.to_datetime(ndf['pubDate'])
ndf['ansDate'] = pd.to_datetime(ndf['ansDate'])


In [ ]:
ndf.sort_values(by=['stock_code', 'pubDate'], inplace=True)
ndf.reset_index(inplace=True, drop=True)
ndf['ans_text'].fillna('n', inplace=True)
ndf.to_excel('/Users/akc/Desktop/PHD/research/NewEnergyAssetPricing2024/dataset/text_data/original_data.xlsx', index=False)
ndf.shape

In [ ]:
# 检查 company_name 的缺失值
ndf.isnull().sum()

# test

In [ ]:
import pandas as pd

In [ ]:
data_path = '/Users/akc/Desktop/PHD/research/NewEnergyAssetPricing2024/dataset/text_data/textWithSimilarity.xlsx'
tdf = pd.read_excel(data_path, converters={'stock_code': str, 'pubDate': pd.to_datetime, 'ansDate': pd.to_datetime})
tdf.head()

In [ ]:
tdf['pubDate'] = pd.to_datetime(tdf['pubDate'])
tdf = tdf[tdf['pubDate']>=pd.to_datetime('2018-01-01')]
tdf.shape